# Chapter 8 — the GPT tokenizer

From *Neural Networks: Zero to Hero — The Textbook*.

Run each cell with **Shift+Enter**. Before you run one, say out loud what you expect it to print; being wrong is the useful part.


## Chapter 8 — the GPT tokenizer

**Video:** 2h13m · [youtu.be/zduSFxRajkE](https://youtu.be/zduSFxRajkE) · **Karpathy's framing:** his least favourite part of working with language models, and the source of a surprising amount of their weirdness.

### The problem

Chapter 7 used one token per character, a vocabulary of 65. Simple, and wasteful: a 1,000-character text becomes 1,000 tokens, and since attention cost grows with the *square* of sequence length, you burn context on nothing.

Real models use **sub-word tokens**: common words are one token, rare words split into pieces.

**Why not whole words?** Vocabulary would be unbounded, misspellings and new words would have no representation, and other languages would be shut out. **Why not characters?** Sequences get too long. Sub-words are the compromise. [standard]

> **Say it to a six-year-old.** The computer cannot read letters. Before it sees anything, a different little program chops the writing into pieces and gives each piece a number, a bit like cutting a sentence into puzzle pieces. Some pieces are whole words, some are half words. The computer only ever sees the numbers. So if you ask it how many letters are in a word, it genuinely cannot see them, the way you cannot count the letters in a picture of a word you glimpsed for a second.

### Foundations you need first

**Unicode.** A standard assigning a number to every character in every writing system, "roughly 150,000 characters across 161 scripts as of right now" [transcript]. `A` is 65, `é` is 233, emoji have their own.

**UTF-8.** A way of writing those numbers as bytes. A **byte** holds a value from 0 to 255. ASCII characters take 1 byte; others take 2 to 4. So any text on earth can be expressed as numbers in 0–255, giving a universal starting vocabulary of exactly 256 tokens with nothing excluded.

**Run it.**

In [ ]:
s = "hello"
print("as unicode code points:", [ord(c) for c in s])
print("as utf-8 bytes:        ", list(s.encode('utf-8')))
print()
s2 = "héllo 😀"
print("characters:", len(s2), "| utf-8 bytes:", len(s2.encode('utf-8')))
print("bytes:", list(s2.encode('utf-8')))

**What you should see:**

**Expected output:**

```
as unicode code points: [104, 101, 108, 108, 111]
as utf-8 bytes:         [104, 101, 108, 108, 111]

characters: 7 | utf-8 bytes: 11
bytes: [104, 195, 169, 108, 108, 111, 32, 240, 159, 152, 128]
```

[verified]

Plain English costs one byte per character. The `é` took 2 bytes and the emoji took 4. **That asymmetry is the seed of everything unfair about tokenization.**

### Byte Pair Encoding, step by step

**BPE** is a compression algorithm from 1994, repurposed for tokenization. [standard]

**The toy version.** Take `aaabdaaabac`, 11 characters, vocabulary of 4.

1. Find the most frequent adjacent pair: `aa`.
2. Invent a new symbol `Z` and replace every occurrence: `ZabdZabac`, now 9 characters, vocabulary 5.
3. Repeat. `ab` becomes `Y`: `ZYdZYac`, 7 characters, vocabulary 6.
4. Continue to your target vocabulary size.

Sequence shrinks, vocabulary grows. That is the entire trade.

**Run it.** The real thing, on real text:

In [ ]:
text = ("Unicode is a standard that assigns a number to every character in every writing "
        "system. UTF-8 is a way of writing those numbers as bytes. Tokenization sits between "
        "raw text and the neural network, and it is responsible for a surprising number of "
        "the strange behaviours that large language models exhibit in practice.") * 4

tokens = list(text.encode("utf-8"))

def get_stats(ids):
    counts = {}
    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair, 0) + 1
    return counts

def merge(ids, pair, idx):
    newids, i = [], 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i+1] == pair[1]:
            newids.append(idx); i += 2
        else:
            newids.append(ids[i]); i += 1
    return newids

stats = get_stats(tokens)
top = sorted(((v, k) for k, v in stats.items()), reverse=True)[:3]
print("most common pairs:", [(bytes(k).decode('utf-8', errors='replace'), v) for v, k in top])

vocab_size = 276          # 256 raw bytes + 20 merges
ids, merges = list(tokens), {}
for i in range(vocab_size - 256):
    stats = get_stats(ids)
    pair = max(stats, key=stats.get)
    ids = merge(ids, pair, 256 + i)
    merges[pair] = 256 + i

print("tokens before:", len(tokens), "| after 20 merges:", len(ids))
print(f"compression ratio: {len(tokens)/len(ids):.2f}X")

vocab = {idx: bytes([idx]) for idx in range(256)}
for (p0, p1), idx in merges.items():
    vocab[idx] = vocab[p0] + vocab[p1]
print("learned tokens:", [vocab[i].decode('utf-8', errors='replace') for i in range(256, 276)])

**What you should see:**

**Expected output:**

```
most common pairs: [('s ', 36), ('e ', 32), (' a', 32)]
tokens before: 1264 | after 20 merges: 916
compression ratio: 1.38X
learned tokens: ['s ', 'e ', ' t', 'er', 'an', 't ', 'in', 's a', ' s', ' th', 'ra', 'and', ' n', 'um', 'umb', 'umber', 'y ', 'ha', 'ri', 'ing']
```

[verified]

**Look at that learned vocabulary, because it is the whole idea made visible.** With no linguistic knowledge whatsoever, counting alone discovered `er`, `an`, `in`, `ing`, `and`, and `th`, which are genuinely the most common letter clusters in English. It then discovered them *compositionally*: `um` became `umb` became `umber`, each built from the previous merge. Nobody supplied a dictionary. This is the same "structure falls out of counting" lesson as Chapter 2's `P(u|q) = 0.69`.

**Run it.** Encoding and decoding, with a round-trip test:

In [ ]:
def decode(ids):
    return b"".join(vocab[i] for i in ids).decode("utf-8", errors="replace")

def encode(text):
    tokens = list(text.encode("utf-8"))
    while len(tokens) >= 2:
        stats = get_stats(tokens)
        pair = min(stats, key=lambda p: merges.get(p, float("inf")))   # earliest-learned merge first
        if pair not in merges:
            break
        tokens = merge(tokens, pair, merges[pair])
    return tokens

s = "the neural network is a standard"
print("encode:", encode(s))
print("roundtrip ok:", decode(encode(s)) == s)

**What you should see:**

**Expected output:**

```
encode: [116, 104, 257, 110, 101, 117, 266, 108, 268, 101, 116, 119, 111, 114, 107, 32, 105, 263, 264, 116, 267, 97, 114, 100]
roundtrip ok: True
```

[verified]

**The merges must be applied in the order they were learned**, which is what `min(..., key=merges.get)` enforces. Apply a later merge before an earlier one and you produce tokens the model has never seen, with no error raised.

**Vocabulary size is a hyperparameter.** GPT-2 used 50,257 tokens; GPT-4 uses roughly 100,000 [transcript]. Larger vocabulary means shorter sequences and more text per token, at the cost of a bigger output layer and fewer training examples per token.

### The payoff: LLM quirks explained

**Run it.** Measure the cost of writing in another language, using OpenAI's real tokenizer:

In [ ]:
import tiktoken                       # pip install tiktoken
enc2 = tiktoken.get_encoding("gpt2")
enc4 = tiktoken.get_encoding("cl100k_base")     # GPT-4's

en = "hello how are you"
hi = "नमस्ते आप कैसे हैं"
print("gpt2 vocab:", enc2.n_vocab, "| gpt4 vocab:", enc4.n_vocab)
print("english:", len(enc2.encode(en)), "tokens")
print("hindi:  ", len(enc2.encode(hi)), "tokens")
print(f"blow-up: {len(enc2.encode(hi))/len(enc2.encode(en)):.1f}x")

**What you should see:**

**Expected output:**

```
gpt2 vocab: 50257 | gpt4 vocab: 100277
english: 4 tokens
hindi:   30 tokens
blow-up: 7.5x
```

[verified]

**7.5 times.** The same sentence, the same meaning. A Hindi speaker pays 7.5× the API cost, fits 7.5× less into the context window, and gets worse results because the model sees their language chopped into meaningless fragments. Karpathy measured about 3× on his example [transcript]; the exact factor depends on the language and the sentence, and the direction is always the same. This is not a policy decision anyone made. It is a side effect of learning merges from a mostly-English corpus.

The rest of the list, each traced to tokenization [transcript]:

- **Spelling and character tasks are hard.** Ask a model to reverse a string or count the letters in a word and it struggles, because a whole word may be one opaque token. It never sees the letters.
- **Arithmetic is erratic.** Numbers get chopped into groups that ignore place value, so `1000000` arrives as `100` + `000` + `0`. Digit structure is destroyed before the model sees it.
- **Trailing whitespace breaks completions.** A prompt ending in a space puts the model in a state its training data rarely contains, because spaces normally attach to the *following* word.
- **SolidGoldMagikarp.** A Reddit username frequent enough in the *tokenizer's* training data to earn its own token, but absent from the *model's* training data. Its embedding stayed at random initialization, and feeding it to GPT-2 produced bizarre, evasive, sometimes hostile output [transcript]. A ghost token: allocated, never trained.

**Run it.** See the spelling problem directly:

In [ ]:
print("--- the same word, with and without a leading space")
for w in [" strawberry", "strawberry"]:
    ids = enc4.encode(w)
    print(f"{w!r:16} -> {len(ids)} token(s): {[enc4.decode([i]) for i in ids]}")

print("--- numbers")
for n in ["127", "128", "1234", "12345", "1000000", "3.14159"]:
    ids = enc4.encode(n)
    print(f"{n:>10} -> {len(ids)} token(s): {[enc4.decode([i]) for i in ids]}")

print("--- long words")
for w in [" hello", " antidisestablishmentarianism"]:
    ids = enc4.encode(w)
    print(f"{w!r:32} -> {len(ids)} token(s): {[enc4.decode([i]) for i in ids]}")

**What you should see:**

**Expected output:**

```
--- the same word, with and without a leading space
' strawberry'    -> 1 token(s): [' strawberry']
'strawberry'     -> 3 token(s): ['str', 'aw', 'berry']
--- numbers
       127 -> 1 token(s): ['127']
       128 -> 1 token(s): ['128']
      1234 -> 2 token(s): ['123', '4']
     12345 -> 2 token(s): ['123', '45']
   1000000 -> 3 token(s): ['100', '000', '0']
   3.14159 -> 4 token(s): ['3', '.', '141', '59']
```

[verified]

**Three separate lessons in one output.**

First, the famous strawberry problem. Written mid-sentence with its space, `strawberry` is **one** token: a single opaque number, with the letters completely invisible. Counting its r's is not a reasoning failure, it is a perception failure. At the start of a line it becomes three chunks, `str` + `aw` + `berry`, which are still not letters.

Second, **the same word tokenizes differently depending on whether a space precedes it.** That is the trailing-whitespace quirk made concrete: end your prompt with a space and every following word arrives in its rarer, fragmented form, which the model saw far less during training.

Third, look at `1000000` becoming `100` + `000` + `0`, and `3.14159` becoming `3` + `.` + `141` + `59`. The digits are grouped in threes from the *left*, which is not how arithmetic works; carrying happens from the right. The model has to do long addition on chunks whose boundaries have nothing to do with place value. That `1234` splits as `123` + `4` while `12345` splits as `123` + `45` means the same digit sits in a different token depending on how long the number is.

**Analogy for the whole chapter.** Tokenization is the sensory organ of a language model. Everything it knows arrived through this filter. Asking why it cannot spell is like asking someone to describe the individual pixels of something they glanced at.

### Variants

- **tiktoken**, OpenAI's fast library, which Karpathy recommends reusing rather than training your own: "if you can reuse the GPT-4 tokens and the vocabulary in your application then that's something you should consider" [transcript].
- **SentencePiece**, Google's library used by Llama and others. It runs BPE on Unicode code points rather than bytes and carries a pile of legacy options. The lecture walks through its quirks, including why it renders spaces as `▁` and why it adds a leading space, admitting "I'm not 100% sure why" on both [transcript].
- **minbpe**, Karpathy's own clean reference implementation, released with the lecture.

> **For the PhD in the room.** BPE is a greedy, deterministic bottom-up merge over a static corpus, and the greediness is the interesting weakness: the merge sequence is fixed at training time, so encoding is not the minimum-token segmentation of a string. Alternatives worth knowing: unigram LM tokenization (Kudo, 2018), which fits a probabilistic vocabulary by EM and supports subword regularization by sampling segmentations at training time, and byte-level fallbacks that guarantee no out-of-vocabulary. The open research direction is removing the stage entirely: MegaByte, byte-latent transformers, and related work operate on raw bytes with a learned patching scheme, motivated by exactly the failures listed above plus the fact that vocabulary is a frozen decision that cannot adapt to a new domain post-training. Also note the compute asymmetry: the embedding and output layers scale with |V|, so GPT-2's 50,257 × 768 embedding is 40M parameters, about 30% of the 124M model [transcript].

### Exercises

1. **Run 500 merges instead of 20** on a megabyte of text and watch the compression ratio climb toward 3×. Print the vocabulary and find where it starts learning whole words.
2. **Tokenize your own name** with `tiktoken` and see whether it is one token or several. Common names are one; unusual ones fragment.
3. **Measure a language you speak** against English with the code above, and compute what the token tax costs at current API prices.
4. **Break the merge order.** Apply merges in reverse order in `encode()` and confirm the round-trip fails, then explain why no error was raised at the point of the mistake.
5. **Train a tokenizer on code** rather than prose, and observe that it learns `):`, `    ` (four spaces), and `self.` as single tokens. This is why code models use code-trained tokenizers.

### Troubleshooting

| Symptom | Cause |
|---|---|
| `UnicodeDecodeError` when decoding | A token boundary split a multi-byte character; use `errors="replace"` |
| Round-trip fails | Merges applied in the wrong order, or a merge missing from the dictionary |
| Compression ratio near 1.0 | Too few merges, or text too short and varied to contain repeated pairs |
| `KeyError` in `vocab[i]` | Encoding produced a token id you never added to the vocabulary |
| tiktoken counts differ from the API's | Different model, hence a different encoding; use `encoding_for_model()` |

### 30-second version

Language models do not read text, they read numbers, and the program that converts one to the other is a compression algorithm that repeatedly merges the most common adjacent pair of symbols. It learns `ing` and `and` and `er` from counting alone, with no linguistic knowledge. It also decides what the model can perceive, which is why models cannot spell (`strawberry` is three chunks: `str`, `aw`, `berry`), why arithmetic is unreliable (`677` is one token, `678` is two), and why the same sentence in Hindi costs 7.5× more than in English.

---